# Identifying a flame's dynamic response in a gas-turbine combustor

A combustor is never a single duct.
Compressor-discharge air enters a plenum and is split among several passages -- the swirler that feeds the flame, the liner-cooling film, and the dilution jets that trim the turbine-inlet temperature -- which recombine before the turbine.
The flame therefore sits **deep inside a branched network**, and the only response one can realistically measure is a *network-wide* one: a transfer matrix between an upstream and a downstream plane.

This notebook builds such a combustor with an **equilibrium flame** and then recovers the flame's dynamic response two ways, from one and the same network measurement:

1. the flame as a **transfer matrix** -- its full linear 2-port, making no assumption about a flame model;
2. the flame **transfer function** (FTF) -- the velocity-driven $n$--$\tau$ feedback, when a model form is assumed.

Both are *de-embedded* from the measured network transfer matrix by the identification layer (`nefes.perturbation.identify`), which knows the rest of the network and solves a per-frequency linear inverse for the unknown element.
We first do it with the full acoustic **and** entropy content, then reduce to the **acoustics-only** workflow that matches how such matrices are actually measured.

In [ ]:
import os, sys
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from thermolib import ThermoInp
from nefes.chem.composition import resolve_composition, enthalpy_mass
from nefes.elements import catalog as cat
from nefes.elements.dynamic_source import n_tau, n_tau_flame
from nefes.shell import Network
from nefes.solver import solve
from nefes.solver.control import states_table
from nefes.thermo.api import EQ_FROZEN, EQ_KERNEL
from nefes.thermo.configure import equilibrium
from nefes.assembly.derive import ES_MDOT, ES_T, ES_P, ES_M, ES_U, ES_RHO
from nefes.perturbation import perturbation_response, TransferMatrix
from nefes.perturbation.identify import (
    identify_transfer_matrix,
    identify_transfer_function,
    UnknownTransferMatrix,
    unknown_dynamic_source,
)

THERMO_INP = os.path.join(_root, "thermolib", "data", "thermo.inp")
lib = ThermoInp(THERMO_INP).library(["O2", "N2", "Ar", "CO2", "CH4", "H2O", "CO", "OH", "H", "O", "NO", "H2"])
AIR = {"O2": 0.2095, "N2": 0.7808, "Ar": 0.0093, "CO2": 0.0004}
FUEL = {"CH4": 1.0}
FREQS = np.linspace(60.0, 900.0, 25)   # measurement frequency grid [Hz]

## The combustor network

Air enters at a single inlet and the **plenum** (a lossless splitter) distributes it among three passages:

* the **swirler** passage, where fuel is injected and the **flame** burns (the primary zone);
* the **liner-cooling** passage, which bypasses the flame and rejoins just downstream;
* the **dilution** passage, which rejoins further down to cool the products to the turbine-inlet temperature.

Everything recombines into a single exit feeding the turbine, so the whole assembly is a genuine two-terminal (inlet/outlet) network with three internal branches.

```
                        premix                       flame-tube
 air-in --[plenum]======[fuel]==[swirler]==( FLAME )============[merge]==
             |                                                    |  (cool)
             |=================[ liner cooling ]==================|
             |                                                             [merge]== exit == turbine-in
             |=========================[ dilution ]========================|
```

The flame is the **equilibrium flame**: the burnt state is the H/P chemical equilibrium of the premixed reactants, so its mean dilatation, temperature rise and composition change are all thermodynamic (no tuned heat-release number).

In [ ]:
# swirler / bypass passage areas set the air split; fuel sets the primary equivalence ratio
A_SW, A_BY, MDOT_AIR, MDOT_FUEL, TAIR, P = 0.045, 0.035, 1.0, 0.020, 700.0, 6.0e5
F_STOICH_CH4 = 0.0583  # stoichiometric CH4/air mass ratio


def build_combustor(dynamic_source=None, plenum=None):
    # branched combustor; returns (prob, x_bar, network).  dynamic_source attaches an FTF to the flame;
    # plenum overrides the air-split element (default: a lossless splitter).
    Yair, _ = resolve_composition(lib, AIR)
    h_ref = enthalpy_mass(lib, Yair, TAIR)
    els = [
        cat.mass_flow_inlet(MDOT_AIR, TAIR, composition=AIR, name="air-in"),          # 0
        plenum if plenum is not None else cat.splitter(name="plenum"),                # 1
        cat.duct(0.15, name="swirler"),                                               # 2
        cat.mass_source(MDOT_FUEL, TAIR, composition=FUEL, name="fuel"),              # 3
        cat.duct(0.15, name="premix"),                                                # 4
        cat.equilibrium_flame(name="flame", dynamic_source=dynamic_source),           # 5  <-- identify this
        cat.duct(0.25, name="flame-tube"),                                            # 6
        cat.junction(name="merge-cool"),                                              # 7
        cat.duct(0.40, name="liner-cool"),                                            # 8
        cat.duct(0.20, name="primary-zone"),                                          # 9
        cat.junction(name="merge-dil"),                                               # 10
        cat.duct(0.60, name="dilution"),                                              # 11
        cat.duct(0.20, name="exit"),                                                  # 12
        cat.pressure_outlet(P, Tt_backflow=TAIR, composition=AIR, name="turbine-in"), # 13
    ]
    edges = [
        (0, 1, A_SW + A_BY),                                # e0  air-in -> plenum
        (1, 2, A_SW), (2, 3, A_SW), (3, 4, A_SW),          # e1-e3 swirler -> fuel -> premix
        (4, 5, A_SW), (5, 6, A_SW), (6, 7, A_SW),          # e4 premix->FLAME  e5 FLAME->tube  e6 ->merge-cool
        (1, 8, A_BY * 0.5), (8, 7, A_BY * 0.5),            # e7,e8 liner cooling -> merge-cool
        (7, 9, A_SW + A_BY * 0.5), (9, 10, A_SW + A_BY * 0.5),  # e9,e10 primary zone -> merge-dil
        (1, 11, A_BY * 0.5), (11, 10, A_BY * 0.5),         # e11,e12 dilution -> merge-dil
        (10, 12, A_SW + A_BY), (12, 13, A_SW + A_BY),      # e13,e14 exit -> turbine-in
    ]
    burnt = {5, 6, 9, 10, 13, 14}  # edges carrying burnt gas -> equilibrium closure; the rest stay frozen (unburnt)
    edge_models = [EQ_KERNEL if e in burnt else EQ_FROZEN for e in range(len(edges))]
    network = Network(equilibrium(lib), p_ref=P, mdot_ref=MDOT_AIR, h_ref=abs(h_ref), nodes=els, edges=edges,
                      edge_models=edge_models)
    prob = cat.build_problem(equilibrium(lib), els, edges, mdot_ref=MDOT_AIR, p_ref=P, h_ref=abs(h_ref),
                             edge_models=edge_models)
    res = solve(prob)
    assert res.converged, "mean flow did not converge"
    return prob, res.x, network


# edge / node bookkeeping used throughout
FLAME_NODE = 5
E_UP, E_DOWN = 4, 5     # the flame's upstream (premix) and downstream (flame-tube) faces
E_IN, E_OUT = 0, 14     # the measurement planes: inlet and turbine-inlet

In [ ]:
prob, x_bar, network = build_combustor()
est = states_table(prob, x_bar)
phi = (MDOT_FUEL / est[ES_MDOT, 1]) / F_STOICH_CH4
print(f"primary zone: swirler air {est[ES_MDOT,1]:.3f} kg/s, phi = {phi:.2f}")
print(f"flame:  premix {est[ES_T,E_UP]:.0f} K  ->  burnt {est[ES_T,E_DOWN]:.0f} K   (rho drop {est[ES_RHO,E_UP]/est[ES_RHO,E_DOWN]:.2f}x)")
print(f"turbine inlet: {est[ES_T,E_OUT]:.0f} K after dilution;  peak Mach {est[ES_M].max():.3f}")
print(f"air split -- swirler {est[ES_MDOT,1]:.2f} | liner {est[ES_MDOT,7]:.2f} | dilution {est[ES_MDOT,11]:.2f} kg/s")

In [ ]:
# axial temperature along the swirler/flame path and the turbine-inlet state
path_edges = [0, 1, 4, 5, 6, 9, 10, 13, 14]
path_names = ["air-in", "swirler", "premix", "flame", "flame-tube", "primary", "merge-dil", "exit", "turbine-in"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=path_names, y=[est[ES_T, e] for e in path_edges],
                         mode="lines+markers", line=dict(color="#c0392b", width=3), marker=dict(size=9)))
fig.update_layout(title="Mean static temperature along the core path",
                  yaxis_title="T [K]", template="plotly_white", height=380)
fig.show()

## The measurement, and the inverse problem

Suppose we can measure the network **transfer matrix** $M(\omega)$ between the inlet plane and the turbine-inlet plane -- the linear map from the wave amplitudes at the inlet edge to those at the outlet edge, $w_\text{out} = M\, w_\text{in}$.
(In practice this is what a multi-microphone / multi-load acoustic characterization returns; here we synthesize it from the model so we have a ground truth to check against.)

Everything in the network *except the flame's own response* is known.
The identification layer condenses that known remainder onto the flame's two faces and solves, at each frequency, a linear inverse for the flame's contribution -- a de-embedding that works for this branched topology exactly as it would for a simple duct in series.

We give the flame a velocity-driven $n$--$\tau$ transfer function as the ground-truth response, then forget it and try to recover it.

In [ ]:
N_GAIN, TAU = 1.0, 3.0e-3                       # ground-truth FTF: gain n, time lag tau [s]
ftf_true = n_tau(N_GAIN, TAU)
prob_ftf, x_ftf, _ = build_combustor(dynamic_source=n_tau_flame(N_GAIN, TAU, ref_edge=E_UP, quantity="u"))

# the "measured" network response, driving the full acoustic + entropy content (a 3x3 matrix)
resp_full = perturbation_response(prob_ftf, x_ftf, FREQS, excite=("acoustic", "entropy"))
M_full = TransferMatrix(FREQS, resp_full.transfer_matrix(E_IN, E_OUT, basis="char"), basis="char")
T_flame_true = resp_full.transfer_matrix(E_UP, E_DOWN, basis="char")   # ground-truth flame 2-port (3x3)
print("measured network matrix:", M_full)

In [ ]:
# plotting helpers (magnitude / phase overlays of true vs identified)
CH = ("f", "g", "h")  # forward acoustic, backward acoustic, entropy


def plot_matrix(freqs, T_true, T_id, title):
    N = T_true.shape[1]
    fig = make_subplots(rows=N, cols=N, subplot_titles=[f"T[{CH[i]},{CH[j]}]" for i in range(N) for j in range(N)])
    for i in range(N):
        for j in range(N):
            r, c = i + 1, j + 1
            fig.add_trace(go.Scatter(x=freqs, y=np.abs(T_true[:, i, j]), mode="lines",
                                     line=dict(color="#2c3e50", width=3), showlegend=(i == 0 and j == 0),
                                     name="true |T|"), row=r, col=c)
            fig.add_trace(go.Scatter(x=freqs, y=np.abs(T_id[:, i, j]), mode="markers",
                                     marker=dict(color="#e67e22", size=5), showlegend=(i == 0 and j == 0),
                                     name="identified"), row=r, col=c)
    fig.update_layout(title=title, template="plotly_white", height=240 * N, width=260 * N)
    fig.update_xaxes(title_text="f [Hz]", row=N)
    fig.update_yaxes(rangemode="tozero")  # magnitudes from 0: a flat (constant) entry then reads as flat, not auto-zoomed to its rounding band
    return fig


def plot_ftf(freqs, F_true, F_id, title):
    fig = make_subplots(rows=1, cols=2, subplot_titles=("gain |F|", "phase [rad]"))
    fig.add_trace(go.Scatter(x=freqs, y=np.abs(F_true), mode="lines", line=dict(color="#2c3e50", width=3),
                             name="true"), row=1, col=1)
    fig.add_trace(go.Scatter(x=freqs, y=np.abs(F_id), mode="markers", marker=dict(color="#e67e22", size=6),
                             name="identified"), row=1, col=1)
    fig.add_trace(go.Scatter(x=freqs, y=np.unwrap(np.angle(F_true)), mode="lines", line=dict(color="#2c3e50", width=3),
                             showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=freqs, y=np.unwrap(np.angle(F_id)), mode="markers", marker=dict(color="#e67e22", size=6),
                             showlegend=False), row=1, col=2)
    fig.update_layout(title=title, template="plotly_white", height=360)
    fig.update_xaxes(title_text="f [Hz]")
    return fig


def rel_err(a, b):
    return np.max(np.abs(a - b)) / np.max(np.abs(b))

## (A) The flame as a transfer matrix -- no model assumed

We mark the flame's acoustic response as an **unknown transfer matrix** and de-embed it from the measured network matrix.
Nothing about a flame model enters: the recovered 3×3 is the flame's full linear 2-port (its passive scattering *and* its active $n$--$\tau$ response, folded into one matrix), in the characteristic basis $(f, g, h)$ = (forward acoustic, backward acoustic, entropy).

In [ ]:
id_TM = identify_transfer_matrix(prob_ftf, x_ftf, M_full, node=FLAME_NODE, a=E_IN, b=E_OUT)
print("identified", id_TM)
print(f"recovery error vs ground-truth flame matrix: {rel_err(id_TM.transfer_matrix(FREQS), T_flame_true):.2e}")
print(f"de-embed conditioning (max over frequency):  {id_TM.conditioning.max():.1e}")
plot_matrix(FREQS, T_flame_true, id_TM.transfer_matrix(FREQS), "Flame transfer matrix (3x3): true vs identified").show()

### Reading the matrix -- where the flame's dynamics live

The recovered matrix is the flame's *full* linear 2-port, so its physics is legible entry by entry.
Two features are worth calling out -- one active, one passive.

**Bottom row $T[h,f]$, $T[h,g]$ -- the entropy the flame sheds.**
An active flame turns a velocity fluctuation into an unsteady heat-release fluctuation, and that hot-/cold-spot convects downstream as an **entropy wave**.
So an *acoustic* input ($f$ or $g$) produces a downstream *entropy* output, and -- because the mechanism is the FTF itself -- these two entries carry the FTF's $e^{-i\omega\tau}$ signature.
Numerically, the active-minus-passive part of the row is proportional to $F(\omega)$ (shape-identical to $\sim\!10^{-14}$, opposite sign for $f$ vs $g$, scaled by a real mean-flow gain).
This is the indirect (entropy) noise the flame injects, and the identification recovers it along with everything else -- you do not model it separately.

**Right column $T[f,h]$, $T[g,h]$ -- the passive response, which can *look* noisy.**
An incoming **entropy** wave carries no velocity fluctuation at the flame, so it does not excite the FTF.
That column is the flame's **passive** scattering, frequency-independent for a compact flame -- a constant.
(Check: with the FTF removed, *every* entry is flat; all the oscillation above is the flame's active response.)
A constant entry has no vertical scale of its own, so an autoscaled plot zooms to the $\sim\!10^{-11}$ floating-point band and magnifies rounding into apparent scatter; drawn from zero (as above) the markers sit on the flat line.

Every entry matches the truth to $\sim\!10^{-11}$ (the printed error).
The lone outlier some entries show near $\sim\!660\;\mathrm{Hz}$ is a network acoustic resonance where the de-embedding is momentarily worse-conditioned (visible in the conditioning figure at the end); it lifts that single point to $\sim\!10^{-10}$, still negligible.

## (B) The flame transfer function -- an $n$--$\tau$ model

If instead we assume the flame responds to the premix **velocity** with a single transfer function, we mark that feedback as unknown (its reference edge and quantity declared) and recover the scalar $F(\omega)$.
The de-embed reports a per-frequency **conditioning** number: it is well posed when the measurement is sensitive to the flame's feedback, and a consistency residual confirms the single-input model fits the data.

In [ ]:
prob_unk, x_unk, _ = build_combustor(dynamic_source=unknown_dynamic_source([(E_UP, "u")]))
id_TF = identify_transfer_function(prob_unk, x_unk, M_full, node=FLAME_NODE, a=E_IN, b=E_OUT)
F_id = id_TF.values[0]
print(f"recovered term: {id_TF.terms[0]}")
print(f"FTF recovery error: {rel_err(F_id, ftf_true(FREQS)):.2e}")
print(f"conditioning <= {id_TF.conditioning.max():.1e},  consistency residual <= {id_TF.residual.max():.1e}")
plot_ftf(FREQS, ftf_true(FREQS), F_id, "Flame transfer function: true n-tau vs identified").show()

## Reducing to the acoustics-only workflow

The full 3×3 above carries the entropy wave the flame sheds.
In practice a combustor 2-port is characterized **acoustically** -- pressure and velocity, not entropy -- and one wants a purely acoustic identification.

Passing `isentropic=True` pins the entropy characteristic to zero everywhere, so the network reduces to its two acoustic waves and the identification returns the classic **2×2 acoustic** flame transfer matrix (and, for the FTF, uses only the acoustic channels).
This is the usual mode; the composition/entropy machinery above is there when you need indirect-noise or entropy content, but the default engineering answer is acoustic.

In [ ]:
# the measured matrix under the same (acoustics-only) assumption -> a 2x2
resp_ac = perturbation_response(prob_ftf, x_ftf, FREQS, excite=("acoustic",), isentropic=True)
M_ac = TransferMatrix(FREQS, resp_ac.transfer_matrix(E_IN, E_OUT, basis="char"), basis="char")
T_flame_ac = resp_ac.transfer_matrix(E_UP, E_DOWN, basis="char")   # 2x2 ground truth

id_TM_ac = identify_transfer_matrix(prob_ftf, x_ftf, M_ac, node=FLAME_NODE, a=E_IN, b=E_OUT, isentropic=True)
id_TF_ac = identify_transfer_function(prob_unk, x_unk, M_ac, node=FLAME_NODE, a=E_IN, b=E_OUT, isentropic=True)
print(f"[acoustic 2x2] matrix recovery error: {rel_err(id_TM_ac.transfer_matrix(FREQS), T_flame_ac):.2e}"
      f"  cond <= {id_TM_ac.conditioning.max():.1e}")
print(f"[acoustic]     FTF recovery error:    {rel_err(id_TF_ac.values[0], ftf_true(FREQS)):.2e}")
plot_matrix(FREQS, T_flame_ac, id_TM_ac.transfer_matrix(FREQS), "Acoustic flame transfer matrix (2x2): true vs identified").show()

## Identifiability, and how the flow-split model matters

The recovered response is exact to rounding because the model is exact and the measurement is clean.
On real data two diagnostics decide whether it can be trusted, and both are returned on every result:

* **Conditioning** of the per-frequency inverse.
  The de-embed solves a small linear system for the unknown's modal coordinates; its condition number is how strongly the *measurement* responds to the *unknown*.
  A well-posed measurement keeps it modest (a few hundred here); a blind or degenerate one sends it to $10^{15}$ and the recovery is meaningless.
* **Consistency** of a transfer-function fit (its residual).
  A single-input FTF that cannot explain the data leaves a finite residual -- the cue to add a term (a pressure-sensitivity term, say) or to fall back to the full transfer-matrix description, which assumes no model.

### The flow-split element is not acoustically innocent

How the plenum divides the air changes what can be identified -- a point worth dwelling on, because the obvious choice is the wrong one.

A **forced flow divider** (`forced_splitter`) prescribes each branch's mass flow as a fixed fraction of the inflow, $\dot m_\text{branch} = \beta\,\dot m_\text{in}$.
That relation is linear and is inherited *unchanged* by the perturbation, so the fluctuation obeys $\dot m_\text{branch}' = \beta\,\dot m_\text{in}'$ at **every** frequency: the branch mass-flow is rigidly **slaved to the inlet**, regardless of what happens on the branch.
It removes an acoustic degree of freedom from the branch.
The flame sits on that branch, so its 2-port can no longer respond independently to the network -- and the sensitivities the de-embed needs collapse, leaving the per-frequency system rank-deficient.
The flame **transfer matrix becomes unidentifiable** from an inlet--outlet measurement (its conditioning explodes), even though the scalar **FTF survives** -- a single transfer function needs only one sensitivity direction, not the full 2-port observability.

The fix is to model the split by **physics rather than decree**: a lossless plenum (`splitter`, shared total pressure, used here) or, more realistically, **resistive passages** -- the swirler and cooling holes each carry a pressure drop set by their effective area (a `loss`/orifice).
Both let the branch flow respond to the acoustics, so observability is restored and the mean split emerges from the geometry instead of being imposed.

The cell below makes this concrete: the same combustor with a *forced* plenum leaves the FTF recoverable but sends the matrix conditioning to the ceiling.

In [ ]:
# same combustor, but a FORCED (constant-fraction) plenum: swirler 40%, liner 30%, dilution 30%
def forced_plenum():
    return cat.forced_splitter([0.40, 0.30], name="plenum")

prob_fs, x_fs, _ = build_combustor(dynamic_source=n_tau_flame(N_GAIN, TAU, ref_edge=E_UP, quantity="u"), plenum=forced_plenum())
M_fs = TransferMatrix(FREQS, perturbation_response(prob_fs, x_fs, FREQS, excite=("acoustic", "entropy")).transfer_matrix(E_IN, E_OUT, basis="char"))
tm_fs = identify_transfer_matrix(prob_fs, x_fs, M_fs, node=FLAME_NODE, a=E_IN, b=E_OUT, continue_=False)

prob_fsu, x_fsu, _ = build_combustor(dynamic_source=unknown_dynamic_source([(E_UP, "u")]), plenum=forced_plenum())
tf_fs = identify_transfer_function(prob_fsu, x_fsu, M_fs, node=FLAME_NODE, a=E_IN, b=E_OUT, continue_=False)

print("plenum model      flame-matrix cond      matrix err        FTF err")
print(f"plain splitter    {id_TM.conditioning.max():.1e}                {rel_err(id_TM.transfer_matrix(FREQS), T_flame_true):.1e}          {rel_err(id_TF.values[0], ftf_true(FREQS)):.1e}")
print(f"forced divider    {tm_fs.conditioning.max():.1e}                {rel_err(tm_fs.transfer_matrix.data, T_flame_true):.1e}          {rel_err(tf_fs.values[0], ftf_true(FREQS)):.1e}")

The continued results (`id_TM.transfer_matrix`, `id_TF.transfer_functions`) are analytic in complex frequency, so they drop straight into the stability eigensolver as a measured-flame closure.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=FREQS, y=id_TM.conditioning, mode="lines+markers", name="flame matrix (3x3), plain split"))
fig.add_trace(go.Scatter(x=FREQS, y=id_TM_ac.conditioning, mode="lines+markers", name="flame matrix (2x2, acoustic)"))
fig.add_trace(go.Scatter(x=FREQS, y=id_TF.conditioning, mode="lines+markers", name="FTF (single input)"))
fig.add_trace(go.Scatter(x=FREQS, y=tm_fs.conditioning, mode="lines+markers", name="flame matrix, FORCED split (singular)"))
fig.update_layout(title="De-embedding conditioning vs frequency", xaxis_title="f [Hz]",
                  yaxis_title="condition number", yaxis_type="log", template="plotly_white", height=400)
fig.show()

## Export for FNetLibUI

The full network is available as `network` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML — `sol.to_yaml(path)` embeds the mean-flow result fields, `network.to_yaml(path)` writes the topology only — then open the file in FNetLibUI.

In [ ]:
import os, tempfile

sol = network.solve()
_out = os.path.join(tempfile.mkdtemp(), "flame_identification.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use network.to_yaml(_out) for topology only
print("saved case:", _out)